# 星系分类网络一键复现

项目数据包含1,314,678源、59字段。默认 **verify** 模式校验现有结果并重绘图表；**recompute** 从本地融合表重新拟合全部分析；**raw** 先从本地原始星表及395个元数据快照重建融合表，再重算分析。两种重算模式强制忽略模型缓存，可能耗时数十分钟。

在项目根目录使用Python 3.13执行 `python -m pip install -r requirements.txt`，再运行 `jupyter lab notebooks/星系分类网络_一键重跑.ipynb`，选择对应环境并运行所有单元。命令行等价入口：`python src/run_local_pipeline.py --mode verify`。

三个模式均只处理本机文件，不调用SSH或远程采集。已归档的原始快照必须保留。

In [ ]:
from pathlib import Path
import json, subprocess, sys
import pandas as pd
from IPython.display import display, Image, Markdown
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
assert (ROOT/'src/run_local_pipeline.py').is_file()
MODE = 'verify'  # 改为 'recompute' 或 'raw' 可强制完整重算
assert MODE in ('verify','recompute','raw')
print('项目:', ROOT, '\nPython:',sys.executable,'\n模式:',MODE)


In [ ]:
# 一键运行，完整命令及阶段耗时写入 results/runs/。
proc = subprocess.run([sys.executable, 'src/run_local_pipeline.py', '--mode', MODE],
                      cwd=ROOT, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
print(proc.stdout)
assert proc.returncode == 0, '请查看上面指向的阶段日志'


In [ ]:
d=json.loads((ROOT/'results/classification/model_comparison.json').read_text())
rows=[]
for name,v in d['cv'].items():
    a=v['aggregate']
    rows.append({'模型':name, **{k:f"{a[k]['mean']:.4f}±{a[k]['std']:.4f}" for k in ['accuracy','macro_f1','macro_pr_auc']}})
display(pd.DataFrame(rows))
display(Markdown('每个外层训练折内独立搜索：30,000源、内层3折。最终在完整外层训练折重训。'))


In [ ]:
# 13幅图；局部SHAP使用未参与对应模型训练的留出源。
figures=sorted((ROOT/'results/figures').glob('fig*.png'))
assert len(figures)>=13
for p in figures:
    display(Markdown('**'+p.name+'**'))
    display(Image(filename=str(p),width=650))


In [ ]:
extra=json.loads((ROOT/'results/evaluation/supplement.json').read_text())
display(pd.read_csv(ROOT/'results/evaluation/error_slices.csv'))
display(pd.DataFrame([{k:x[k] for k in ['source_id','true','pred','expected_probability','shap_sum','probability']} for x in extra['local_samples']]))
validation=json.loads((ROOT/'results/review_20261116/submission_validation.json').read_text())
assert validation['passed']
print('独立验证检查项:',validation['check_count'])
print('仍待填写:',validation['team_facts'])


## 结果与边界

主报告Markdown位于 `docs/期末报告_星系分类网络.md`。DOCX/PDF打印文件另经模板排版及逐页渲染校验。

本项目使用测光特征分类，图像元数据用于融合验收与追溯。没有像素分类训练，没有新的独立测试集，未做天区分块；模拟缺失不证明真实缺失机制。组员事实和真实沟通记录须按实际情况补入。